# Evaluating Random Forest

In this final exercise you'll be evaluating the results of cross-validation on a Random Forest model.

The following have already been created:

- `cv` - a cross-validator which has already been fit to the training data
- `evaluator` — a `BinaryClassificationEvaluator` object and
- `flights_test` — the testing data.

## Instructions

- Print a list of average AUC metrics across all models in the parameter grid.
- Display the average AUC for the best model. This will be the largest AUC in the list.
- Print an explanation of the `maxDepth` and `featureSubsetStrategy` parameters for the best model.
- Display the AUC for the best model predictions on the testing data.

In [1]:
# # Import the SparkSession class
# import pyspark
# from pyspark.sql import SparkSession

# spark = SparkSession.builder.appName('flight_manipulate_columns').getOrCreate()


In [1]:
# Intialization
import os
import sys

os.environ["SPARK_HOME"] = "/home/talentum/spark"
os.environ["PYLIB"] = os.environ["SPARK_HOME"] + "/python/lib"
# In below two lines, use /usr/bin/python2.7 if you want to use Python 2
os.environ["PYSPARK_PYTHON"] = "/usr/bin/python3.6" 
os.environ["PYSPARK_DRIVER_PYTHON"] = "/usr/bin/python3"
sys.path.insert(0, os.environ["PYLIB"] +"/py4j-0.10.7-src.zip")
sys.path.insert(0, os.environ["PYLIB"] +"/pyspark.zip")

# NOTE: Whichever package you want mention here.
# os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages com.databricks:spark-xml_2.11:0.6.0 pyspark-shell' 
# os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages org.apache.spark:spark-avro_2.11:2.4.0 pyspark-shell'
os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages com.databricks:spark-xml_2.11:0.6.0,org.apache.spark:spark-avro_2.11:2.4.3 pyspark-shell'
# os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages com.databricks:spark-xml_2.11:0.6.0,org.apache.spark:spark-avro_2.11:2.4.0 pyspark-shell'

In [3]:
#Entrypoint 2.x
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Spark SQL basic example").enableHiveSupport().getOrCreate()

# On yarn:
# spark = SparkSession.builder.appName("Spark SQL basic example").enableHiveSupport().master("yarn").getOrCreate()
# specify .master("yarn")

sc = spark.sparkContext

In [5]:
flights = spark.read.csv('file:///home/talentum/test-jupyter/c5-MLWithPySpark/M4-EnsemblesAndPipelines/4_Ensembles/dataset/flights.csv',
                         sep=',',
						 header=True,
						 inferSchema=True,
						 nullValue='NA')


flights = flights.drop('flight')
flights = flights.dropna()

from pyspark.sql.functions import round
flights = flights.withColumn('km', round(flights.mile * 1.60934, 0))\
                .drop('mile')\
				.withColumn('label', (flights.delay > 15).cast('integer'))

#flights = flights.sample(0.25, seed=13)

from pyspark.ml.feature import VectorAssembler
assembler = VectorAssembler(inputCols=[
    'mon', 'depart', 'duration'
	], outputCol='features')
flights = assembler.transform(flights)
flights = flights.select('mon', 'depart', 'duration', 'features', 'label')

flights_train, flights_test = flights.randomSplit([0.8, 0.2], seed=17)

from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

In [ ]:
# Create a random forest classifier
forest = RandomForestClassifier()

# Create a parameter grid
params = ParamGridBuilder() \
            .addGrid(forest.featureSubsetStrategy, ['all', 'onethird', 'sqrt', 'log2']) \
            .addGrid(forest.maxDepth, [2, 5, 10]) \
            .build()

# Create a binary classification evaluator
evaluator = BinaryClassificationEvaluator()

# Create a cross-validator
cv = CrossValidator(estimator = forest\
                    , estimatorParamMaps = params, evaluator = evaluator, numFolds = 5)
cv = cv.fit(flights_train)

In [ ]:
# from os import system
# import pickle

# # system('unzip ./flights-random-forest-cv.zip')
# from pyspark.ml.tuning import CrossValidatorModel
# # cv = CrossValidatorModel.load('./flights-random-forest-cv')
# cv = CrossValidatorModel.load('E:\\Spark_Hadoop_Windows\\C-MachineLearningWithPySpark\\M4\\4_Ensembles\\flights-random-forest-cv')

# cv.avgMetrics = pickle.load(open('./flights-random-forest-cv-metrics.pickle', 'rb'))

# # from pyspark.ml.evaluation import BinaryClassificationEvaluator
# evaluator = BinaryClassificationEvaluator()

In [ ]:
# # Run this paragraph on Ubuntu

# from os import system
# import pickle

# # system('unzip ./flights-random-forest-cv.zip')
# from pyspark.ml.tuning import CrossValidatorModel
# # cv = CrossValidatorModel.load('./flights-random-forest-cv')
# cv = CrossValidatorModel.load('file:///home/talentum/test-jupyter/C-MachineLearningWithPySpark/M4/4_Ensembles/flights-random-forest-cv')

# cv.avgMetrics = pickle.load(open('file:///home/talentum/test-jupyter/C-MachineLearningWithPySpark/M4/4_Ensembles/flights-random-forest-cv-metrics.pickle', 'rb'))

# # from pyspark.ml.evaluation import BinaryClassificationEvaluator
# evaluator = BinaryClassificationEvaluator()

In [ ]:
# Average AUC for each parameter combination in grid
print(cv.____)

# Average AUC for the best model
print(____(____))

# What's the optimal parameter value for maxDepth?
print(cv.____.explainParam('____'))
# What's the optimal parameter value for featureSubsetStrategy?
print(cv.____.____(____))

# AUC for best model on testing data
print(evaluator.____(____.____(____)))